In [20]:
!pip install -q kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
import sqlite3
import pandas as pd
from IPython.display import display

# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
print("Libraries ready, DB connection open")

Libraries ready, DB connection open


In [21]:
#Import the Superstore dataset into a table named superstore_raw.  
from kagglehub import KaggleDatasetAdapter
import kagglehub

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "vivek468/superstore-dataset-final",
    path="Sample - Superstore.csv",   # ← explicit filename required
    pandas_kwargs={"encoding": "latin-1"}
)

# Normalise column names → snake_case
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_", regex=False)
              .str.replace("-", "_", regex=False))

# Write to SQLite as superstore_raw
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)

print("superstore_raw loaded:", len(df), "rows,", len(df.columns), "columns")
print("Columns:", list(df.columns))
display(df.head(3))

superstore_raw loaded: 9994 rows, 21 columns
Columns: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [22]:
#Create these 3 tables from it:  
#customers  
#orders  
#products

#Insert data into these tables using SELECT DISTINCT.  

sql_statements = [
    """
    CREATE TABLE IF NOT EXISTS customers AS
    SELECT DISTINCT
        customer_id,
        customer_name,
        segment,
        country, city, state,
        postal_code,
        region
    FROM superstore_raw
    """,
    """
    CREATE TABLE IF NOT EXISTS orders AS
    SELECT DISTINCT
        order_id,
        order_date,
        ship_date,
        ship_mode,
        customer_id,
        product_id,
        CAST(sales    AS REAL) AS sales,
        CAST(quantity AS INT)  AS quantity,
        CAST(discount AS REAL) AS discount,
        CAST(profit   AS REAL) AS profit
    FROM superstore_raw
    """,
    """
    CREATE TABLE IF NOT EXISTS products AS
    SELECT DISTINCT
        product_id,
        product_name,
        category,
        sub_category
    FROM superstore_raw
    """
]

for stmt in sql_statements:
    cursor.execute(stmt)
conn.commit()

for tbl in ["customers", "orders", "products"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {tbl}", conn).iloc[0, 0]
    print(f"  {tbl:<12} → {n:,} rows")

print("All 3 tables created")

  customers    → 4,910 rows
  orders       → 9,993 rows
  products     → 1,894 rows
All 3 tables created


In [23]:
#Find all orders where sales are greater than the average sales. (Subquery)  
q1 = """
SELECT order_id,
       customer_id,
       product_id,
       ROUND(sales, 2) AS sales
FROM   orders
WHERE  sales > (SELECT AVG(sales) FROM orders)
ORDER  BY sales DESC
LIMIT  10
"""
result = pd.read_sql(q1, conn)
print(f"Orders above average sales (showing top 10):")
display(result)

Orders above average sales (showing top 10):


,order_id,customer_id,product_id,sales
0,CA-2014-145317,SM-20320,TEC-MA-10002412,22638.48
1,CA-2016-118689,TC-20980,TEC-CO-10004722,17499.95
2,CA-2017-140151,RB-19360,TEC-CO-10004722,13999.96
3,CA-2017-127180,TA-21385,TEC-CO-10004722,11199.97
4,CA-2017-166709,HL-15040,TEC-CO-10004722,10499.97
5,CA-2016-117121,AB-10105,OFF-BI-10000545,9892.74
6,CA-2014-116904,SC-20095,OFF-BI-10001120,9449.95
7,US-2016-107440,BS-11365,TEC-MA-10001047,9099.93
8,CA-2016-158841,SE-20110,TEC-MA-10001127,8749.95
9,CA-2016-143714,CC-12370,TEC-CO-10004722,8399.98


In [24]:
#Find the highest sales order for each customer. (Subquery)  
q2 = """
SELECT o.customer_id,
       c.customer_name,
       o.order_id,
       ROUND(o.sales, 2) AS sales
FROM   orders    o
JOIN   customers c USING (customer_id)
WHERE  o.sales = (
           SELECT MAX(o2.sales)
           FROM   orders o2
           WHERE  o2.customer_id = o.customer_id
       )
ORDER  BY o.sales DESC
LIMIT  10
"""
result = pd.read_sql(q2, conn)
print("Highest-sales order per customer (top 10):")
display(result)

Highest-sales order per customer (top 10):


,customer_id,customer_name,order_id,sales
0,SM-20320,Sean Miller,CA-2014-145317,22638.48
1,SM-20320,Sean Miller,CA-2014-145317,22638.48
2,SM-20320,Sean Miller,CA-2014-145317,22638.48
3,SM-20320,Sean Miller,CA-2014-145317,22638.48
4,SM-20320,Sean Miller,CA-2014-145317,22638.48
5,TC-20980,Tamara Chand,CA-2016-118689,17499.95
6,TC-20980,Tamara Chand,CA-2016-118689,17499.95
7,TC-20980,Tamara Chand,CA-2016-118689,17499.95
8,TC-20980,Tamara Chand,CA-2016-118689,17499.95
9,TC-20980,Tamara Chand,CA-2016-118689,17499.95


In [26]:
#Calculate total sales for each customer. (CTE)  
q3 = """
WITH customer_sales AS (
    SELECT o.customer_id,
           c.customer_name,
           SUM(o.sales) AS total_sales
    FROM   orders    o
    JOIN   customers c USING (customer_id)
    GROUP  BY o.customer_id, c.customer_name
)
SELECT customer_id,
       customer_name,
       ROUND(total_sales, 2) AS total_sales
FROM   customer_sales
ORDER  BY total_sales DESC
LIMIT  10
"""
result = pd.read_sql(q3, conn)
print("Total sales per customer (top 10):")
display(result)

Total sales per customer (top 10):


,customer_id,customer_name,total_sales
0,KL-16645,Ken Lonsdale,155927.52
1,SE-20110,Sanjit Engle,134303.82
2,CL-12565,Clay Ludtke,130566.55
3,AB-10105,Adrian Barton,130262.14
4,SC-20095,Sanjit Chand,127281.01
5,SM-20320,Sean Miller,125215.25
6,EH-13765,Edward Hooks,123730.56
7,GT-14710,Greg Tran,118201.20
8,SV-20365,Seth Vernon,114709.50
9,JL-15835,John Lee,107799.15


In [27]:
#Find customers whose total sales are above average. (CTE + Subquery)  
q4 = """
WITH customer_sales AS (
    SELECT o.customer_id,
           c.customer_name,
           SUM(o.sales) AS total_sales
    FROM   orders    o
    JOIN   customers c USING (customer_id)
    GROUP  BY o.customer_id, c.customer_name
)
SELECT customer_id,
       customer_name,
       ROUND(total_sales, 2) AS total_sales
FROM   customer_sales
WHERE  total_sales > (SELECT AVG(total_sales) FROM customer_sales)
ORDER  BY total_sales DESC
"""
result = pd.read_sql(q4, conn)
print(f"Customers above average total sales: {len(result)}")
display(result.head(10))

Customers above average total sales: 280


,customer_id,customer_name,total_sales
0,KL-16645,Ken Lonsdale,155927.52
1,SE-20110,Sanjit Engle,134303.82
2,CL-12565,Clay Ludtke,130566.55
3,AB-10105,Adrian Barton,130262.14
4,SC-20095,Sanjit Chand,127281.01
5,SM-20320,Sean Miller,125215.25
6,EH-13765,Edward Hooks,123730.56
7,GT-14710,Greg Tran,118201.20
8,SV-20365,Seth Vernon,114709.50
9,JL-15835,John Lee,107799.15


In [28]:
#Rank all customers based on total sales. (Window Function)  
q5 = """
WITH customer_sales AS (
    SELECT o.customer_id,
           c.customer_name,
           SUM(o.sales) AS total_sales
    FROM   orders    o
    JOIN   customers c USING (customer_id)
    GROUP  BY o.customer_id, c.customer_name
)
SELECT customer_id,
       customer_name,
       ROUND(total_sales, 2)                           AS total_sales,
       RANK()       OVER (ORDER BY total_sales DESC)   AS sales_rank,
       DENSE_RANK() OVER (ORDER BY total_sales DESC)   AS dense_rank
FROM   customer_sales
ORDER  BY sales_rank
LIMIT  10
"""
result = pd.read_sql(q5, conn)
print("Customer rankings (top 10):")
display(result)

Customer rankings (top 10):


,customer_id,customer_name,total_sales,sales_rank,dense_rank
0,KL-16645,Ken Lonsdale,155927.52,1,1
1,SE-20110,Sanjit Engle,134303.82,2,2
2,CL-12565,Clay Ludtke,130566.55,3,3
3,AB-10105,Adrian Barton,130262.14,4,4
4,SC-20095,Sanjit Chand,127281.01,5,5
5,SM-20320,Sean Miller,125215.25,6,6
6,EH-13765,Edward Hooks,123730.56,7,7
7,GT-14710,Greg Tran,118201.20,8,8
8,SV-20365,Seth Vernon,114709.50,9,9
9,JL-15835,John Lee,107799.15,10,10


In [29]:
#Assign row numbers to each order within a customer. (Window Function + PARTITION BY)  
q6 = """
SELECT o.customer_id,
       c.customer_name,
       o.order_id,
       o.order_date,
       ROUND(o.sales, 2) AS sales,
       ROW_NUMBER() OVER (
           PARTITION BY o.customer_id
           ORDER BY     o.order_date, o.order_id
       ) AS order_row_num
FROM   orders    o
JOIN   customers c USING (customer_id)
ORDER  BY o.customer_id, order_row_num
LIMIT  15
"""
result = pd.read_sql(q6, conn)
print("Row numbers per order within each customer (first 15 rows):")
display(result)

Row numbers per order within each customer (first 15 rows):


,customer_id,customer_name,order_id,order_date,sales,order_row_num
0,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,1
1,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,2
2,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,3
3,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,4
4,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,5
5,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,6
6,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,7
7,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,8
8,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,9
9,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,10


In [30]:
#Display top 3 customers based on total sales. (Window Function)  
q7 = """
WITH customer_sales AS (
    SELECT o.customer_id,
           c.customer_name,
           SUM(o.sales) AS total_sales
    FROM   orders    o
    JOIN   customers c USING (customer_id)
    GROUP  BY o.customer_id, c.customer_name
),
ranked AS (
    SELECT customer_id,
           customer_name,
           ROUND(total_sales, 2)                          AS total_sales,
           DENSE_RANK() OVER (ORDER BY total_sales DESC)  AS rnk
    FROM   customer_sales
)
SELECT customer_id, customer_name, total_sales, rnk
FROM   ranked
WHERE  rnk <= 3
"""
result = pd.read_sql(q7, conn)
print("Top 3 customers:")
display(result)

Top 3 customers:


,customer_id,customer_name,total_sales,rnk
0,KL-16645,Ken Lonsdale,155927.52,1
1,SE-20110,Sanjit Engle,134303.82,2
2,CL-12565,Clay Ludtke,130566.55,3


In [31]:
#Write one final query that shows: 
#Customer Name  
#Total Sales  
#Rank  
#(Use JOIN + CTE + Window Function together) 

final_q = """
WITH customer_sales AS (
    SELECT o.customer_id,
           SUM(o.sales) AS total_sales
    FROM   orders o
    GROUP  BY o.customer_id
),
ranked_customers AS (
    SELECT cs.customer_id,
           c.customer_name,
           ROUND(cs.total_sales, 2)                     AS total_sales,
           RANK() OVER (ORDER BY cs.total_sales DESC)   AS rank
    FROM   customer_sales cs
    JOIN   customers c USING (customer_id)
)
SELECT customer_name,
       total_sales,
       rank
FROM   ranked_customers
ORDER  BY rank
"""
result = pd.read_sql(final_q, conn)
print("Final combined query — all customers ranked:")
display(result)

Final combined query — all customers ranked:


,customer_name,total_sales,rank
0,Sean Miller,25043.05,1
1,Sean Miller,25043.05,1
2,Sean Miller,25043.05,1
3,Sean Miller,25043.05,1
4,Sean Miller,25043.05,1
...,...,...,...
4905,Mitch Gastineau,16.74,4906
4906,Carl Jackson,16.52,4907
4907,Lela Donovan,5.30,4908
4908,Thais Sissman,4.83,4909


In [39]:
#Mini Project: Customer Sales Insights 
#Who are the top 5 customers?  
mp1 = """
WITH cs AS (
    SELECT o.customer_id, c.customer_name,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM   orders o JOIN customers c USING (customer_id)
    GROUP  BY o.customer_id, c.customer_name
),
r AS (
    SELECT *, DENSE_RANK() OVER (ORDER BY total_sales DESC) AS rnk
    FROM cs
)
SELECT customer_id, customer_name, total_sales, rnk
FROM   r
WHERE  rnk <= 5
"""
print("── Top 5 Customers ──────────────────────")
display(pd.read_sql(mp1, conn))


── Top 5 Customers ──────────────────────


,customer_id,customer_name,total_sales,rnk
0,KL-16645,Ken Lonsdale,155927.52,1
1,SE-20110,Sanjit Engle,134303.82,2
2,CL-12565,Clay Ludtke,130566.55,3
3,AB-10105,Adrian Barton,130262.14,4
4,SC-20095,Sanjit Chand,127281.01,5


In [40]:
#Who are the bottom 5 customers?  
mp2 = """
WITH cs AS (
    SELECT o.customer_id, c.customer_name,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM   orders o JOIN customers c USING (customer_id)
    GROUP  BY o.customer_id, c.customer_name
),
r AS (
    SELECT *, DENSE_RANK() OVER (ORDER BY total_sales ASC) AS rnk
    FROM cs
)
SELECT customer_id, customer_name, total_sales, rnk
FROM   r
WHERE  rnk <= 5
"""

print("── Bottom 5 Customers ───────────────────")
display(pd.read_sql(mp2, conn))

── Bottom 5 Customers ───────────────────


,customer_id,customer_name,total_sales,rnk
0,LD-16855,Lela Donovan,5.30,1
1,TS-21085,Thais Sissman,9.67,2
2,CJ-11875,Carl Jackson,16.52,3
3,MG-18205,Mitch Gastineau,16.74,4
4,RS-19870,Roy Skaria,44.66,5


In [41]:
#Which customers made only one order?  
mp3 = """
SELECT o.customer_id,
       c.customer_name,
       COUNT(DISTINCT o.order_id) AS order_count
FROM   orders    o
JOIN   customers c USING (customer_id)
GROUP  BY o.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1
ORDER  BY customer_name
"""
print("── Customers who placed only ONE order ──")
r3 = pd.read_sql(mp3, conn)
print(f"   Found: {len(r3)} customers")
display(r3)


── Customers who placed only ONE order ──
   Found: 12 customers


,customer_id,customer_name,order_count
0,AR-10570,Anemone Ratner,1
1,AO-10810,Anthony O'Donnell,1
2,CJ-11875,Carl Jackson,1
3,JC-15385,Jenna Caffey,1
4,JR-15700,Jocasta Rupert,1
5,LD-16855,Lela Donovan,1
6,MG-18205,Mitch Gastineau,1
7,PH-18790,Patricia Hirasaki,1
8,RE-19405,Ricardo Emerson,1
9,RM-19750,Roland Murray,1


In [42]:
#Which customers have above-average sales?  
mp4 = """
WITH cs AS (
    SELECT o.customer_id, c.customer_name,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM   orders o JOIN customers c USING (customer_id)
    GROUP  BY o.customer_id, c.customer_name
)
SELECT customer_id, customer_name, total_sales
FROM   cs
WHERE  total_sales > (SELECT AVG(total_sales) FROM cs)
ORDER  BY total_sales DESC
"""
print("── Customers above average total sales ──")
r4 = pd.read_sql(mp4, conn)
print(f"   Found: {len(r4)} customers")
display(r4.head(10))

── Customers above average total sales ──
   Found: 280 customers


,customer_id,customer_name,total_sales
0,KL-16645,Ken Lonsdale,155927.52
1,SE-20110,Sanjit Engle,134303.82
2,CL-12565,Clay Ludtke,130566.55
3,AB-10105,Adrian Barton,130262.14
4,SC-20095,Sanjit Chand,127281.01
5,SM-20320,Sean Miller,125215.25
6,EH-13765,Edward Hooks,123730.56
7,GT-14710,Greg Tran,118201.20
8,SV-20365,Seth Vernon,114709.50
9,JL-15835,John Lee,107799.15


In [43]:
#What is the highest order value per customer? 
mp5 = """
SELECT o.customer_id,
       c.customer_name,
       o.order_id,
       ROUND(o.sales, 2) AS highest_order_sales
FROM   orders    o
JOIN   customers c USING (customer_id)
WHERE  o.sales = (
           SELECT MAX(o2.sales)
           FROM   orders o2
           WHERE  o2.customer_id = o.customer_id
       )
ORDER  BY highest_order_sales DESC
LIMIT  10
"""
result = pd.read_sql(mp5, conn)
print("Highest order value per customer (top 10):")
display(result)

Highest order value per customer (top 10):


,customer_id,customer_name,order_id,highest_order_sales
0,SM-20320,Sean Miller,CA-2014-145317,22638.48
1,SM-20320,Sean Miller,CA-2014-145317,22638.48
2,SM-20320,Sean Miller,CA-2014-145317,22638.48
3,SM-20320,Sean Miller,CA-2014-145317,22638.48
4,SM-20320,Sean Miller,CA-2014-145317,22638.48
5,TC-20980,Tamara Chand,CA-2016-118689,17499.95
6,TC-20980,Tamara Chand,CA-2016-118689,17499.95
7,TC-20980,Tamara Chand,CA-2016-118689,17499.95
8,TC-20980,Tamara Chand,CA-2016-118689,17499.95
9,TC-20980,Tamara Chand,CA-2016-118689,17499.95
